# Pipeline completo final - TCC Insegurança Alimentar (CadÚnico/AL)

Notebook único consolidando: harmonização → pré-processamento →
desenvolvimento do modelo com tuning → predição de risco (domiciliar e
municipal) → validações de robustez espaço-temporal (E2-E12).

Substitui a fragmentação anterior em 3 notebooks
(`01_preprocess_and_baselines.ipynb`, `02_model_comparison_and_final.ipynb`,
`03_explainability_and_ablation.ipynb`) + 11 scripts `E*.py` standalone,
que **permanecem no repositório como fonte de migração** (não foram
removidos nem editados nesta consolidação).

Ver `desenvolvimento/docs/plano_tabelas_e_notebook_final.md` (Parte B)
para o plano completo. Hiperparâmetros do LightGBM são reuso do tuning
de Barbosa Costa 2026 (`CRIA-main/02_model_comparison_and_final.ipynb`
cell 19), documentado no Apêndice A do TCC.

In [ ]:
import sys
import json
import time
import gc
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path("scripts").resolve()))
import common_pipeline as cp

RANDOM_STATE = cp.RANDOM_STATE
THRESHOLD = cp.THRESHOLD
BEST_LGBM_PARAMS = cp.BEST_LGBM_PARAMS

IN_COLAB = cp.in_colab()
BASE_PATH = cp.get_base_path()  # Drive/tcc_cadunico no Colab; "." localmente
DATA_DIR = BASE_PATH / "data/anon_outputs"
NB1_DIR = BASE_PATH / "outputs/notebook1"
OUT_DIR = BASE_PATH / "outputs/notebook_final"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Ambiente:", "Colab" if IN_COLAB else "local", "| BASE_PATH:", BASE_PATH.resolve())
print("common_pipeline carregado de", Path("scripts/common_pipeline.py").resolve() if not IN_COLAB else "scripts/common_pipeline.py")


## MLflow (sqlite backend, mesmo padrão dos scripts `E*.py`)

In [ ]:
import mlflow
import mlflow.lightgbm

MLRUNS_DIR = Path("mlruns")
MLRUNS_DIR.mkdir(exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{MLRUNS_DIR.as_posix()}/mlflow.db")
mlflow.set_experiment("notebook_final_pipeline_completo")

## Parte 1 — Harmonização e pré-processamento

**Nível de predição: N/A (preparação de dados).**

A harmonização a partir dos CSVs brutos anuais com PII (`tab_cad_*.csv`, código da co-orientadora em `CRIA-main/limpezaDados.ipynb` / `CadUnico-main/limpezaDados.ipynb`) **não é reexecutada aqui** — não faz parte deste repositório, é específica de cada máquina/Drive e envolve dados sensíveis (LGPD) que não devem circular fora do ambiente de quem já tem acesso autorizado. A célula seguinte documenta essa etapa só como referência de proveniência (comentada, não executada).

O ponto de partida real deste notebook é `cadunico_2023_2025_union_anon.csv` (já anonimizado/harmonizado): se os pickles de `outputs/notebook1/` já existirem (execução local anterior), eles são reaproveitados; caso contrário (primeira execução, ou no Google Colab), o pré-processamento completo de `01_preprocess_and_baselines.ipynb` é executado a partir do CSV via `common_pipeline.preprocess_union_csv` e os pickles são salvos para reuso.


In [ ]:
# REFERÊNCIA DE PROVENIÊNCIA — NÃO EXECUTAR.
# Lógica de harmonização (código da co-orientadora, CRIA-main/limpezaDados.ipynb
# e CadUnico-main/limpezaDados.ipynb), reproduzida aqui só para registro de
# proveniência de `cadunico_2023_2025_union_anon.csv`:
#
# RAW_FILES = {
#     "2023": "tab_cad_Dez2023.csv",
#     "2024": "tab_cad_Dez2024.csv",
#     "2025": "tab_cad_Junho2025.csv",
# }
# 1. Le cada CSV bruto anual (chunked, ~2M linhas/ano, contem PII).
# 2. Remove ~50 colunas de identificacao direta (nome, CPF, NIS, endereco).
# 3. Harmoniza por UNIAO de colunas entre os 3 anos (NA onde a coluna nao existe).
# 4. Cria a coluna `ano` e o indicador binario `inseg_alim_bin` a partir de
#    `d.ind_risco_scl_inseg_alim` ({1.0: 1, 2.0: 0}).
# 5. Grava `data/anon_outputs/cadunico_2023_2025_union_anon.csv` (insumo da
#    celula seguinte) e os 3 CSVs anuais `cadunico_{ano}_anon.csv` (usados
#    pelos experimentos E9/E10/E12 mais abaixo, que leem schema/prevalencia
#    por ano diretamente desses arquivos).
#
# Não reexecutado neste notebook: os CSVs brutos com PII não fazem parte
# deste repositório nem deste ambiente de execução.


In [ ]:
UNION_CSV = DATA_DIR / "cadunico_2023_2025_union_anon.csv"

if NB1_DIR.exists() and (NB1_DIR / "X_trainval_prepared.pkl").exists():
    artifacts = cp.load_notebook1_artifacts(NB1_DIR)
    print("Artefatos carregados do cache:", NB1_DIR.resolve())
else:
    assert UNION_CSV.exists(), (
        f"{UNION_CSV} nao encontrado. Coloque o CSV anonimizado "
        "(harmonizado pelo limpezaDados.ipynb da co-orientadora) nesse "
        "path antes de rodar esta celula."
    )
    print("Cache nao encontrado - rodando pre-processamento completo a partir de", UNION_CSV)
    artifacts = cp.preprocess_union_csv(UNION_CSV, save_to=NB1_DIR)
    print("Artefatos gerados e cacheados em:", NB1_DIR.resolve())

X_train, X_val = artifacts["X_train"], artifacts["X_val"]
y_train, y_val = artifacts["y_train"], artifacts["y_val"]
X_trainval, X_test = artifacts["X_trainval"], artifacts["X_test"]
y_trainval, y_test = artifacts["y_trainval"], artifacts["y_test"]
rare_maps, clip_bounds = artifacts["rare_maps"], artifacts["clip_bounds"]

print(f"X_train={X_train.shape}  X_val={X_val.shape}")
print(f"X_trainval={X_trainval.shape}  X_test={X_test.shape}")
print(f"Prevalencia treino={y_train.mean():.4f}  val={y_val.mean():.4f}  "
      f"trainval={y_trainval.mean():.4f}  test={y_test.mean():.4f}")


## Parte 2 — Desenvolvimento do modelo com tuning

**Nível de predição: domiciliar.**

Reproduz a comparação LightGBM `unweighted`/`weighted` × CatBoost (E7)
e retreina o modelo final sobre `X_trainval` para avaliação no teste
2025 (E1). Hiperparâmetros reusados do tuning de Barbosa Costa 2026.

In [ ]:
X_full_common = cp.apply_clip_bounds(cp.apply_rare_maps(X_trainval, rare_maps), clip_bounds)
X_test_common = cp.apply_clip_bounds(cp.apply_rare_maps(X_test, rare_maps), clip_bounds)

common_drop_cols = [c for c in cp.DROP_HIGH_CARD_TEXT if c in X_full_common.columns]
X_full_common = X_full_common.drop(columns=common_drop_cols, errors="ignore")
X_test_common = X_test_common.drop(columns=common_drop_cols, errors="ignore")
X_test_common = X_test_common.reindex(columns=X_full_common.columns)

print(f"X_full_common: {X_full_common.shape}  (drop de {len(common_drop_cols)} colunas high-card text)")

In [ ]:
X_tr_prep = cp.prepare_lgbm_input(cp.cast_numeric_light(X_full_common))
X_te_prep = cp.prepare_lgbm_input(X_test_common, reference_columns=X_tr_prep.columns.tolist())

t0 = time.time()
with mlflow.start_run(run_name="E1_final_model"):
    final_model = cp.train_lgbm(X_tr_prep, y_trainval, scale_pos_weight=1.0)
    y_proba_test = final_model.predict_proba(X_te_prep)[:, 1]
    metrics_e1 = cp.evaluate(y_test, y_proba_test)
    for k, v in metrics_e1.items():
        mlflow.log_metric(k.replace("-", "_"), v)
print(f"Treino+avaliacao em {time.time()-t0:.1f}s")
print("E1 (reproducao do pipeline principal):")
for k, v in metrics_e1.items():
    print(f"  {k}: {v:.4f}")
print("\nEsperado (Barbosa2026EN): AUC-ROC=0.9083  AUC-PR=0.7232  Brier=0.0488")
print("Esperado (este trabalho, ja documentado em Cap05 tab:e1_reproducao): "
      "AUC-ROC=0.9066  AUC-PR=0.7174  Brier=0.0496")

joblib.dump(final_model, OUT_DIR / "final_model.pkl")

### Predições individuais (domiciliar) salvas para reuso nas Partes 3 e 4

In [ ]:
df_pred_2025 = pd.DataFrame({
    "cd_ibge": X_test_common["d.cd_ibge"].astype(int).values,
    "y_true": y_test.values,
    "y_proba": y_proba_test,
})
df_pred_2025.to_csv(OUT_DIR / "predicoes_individuais_2025.csv", index=False)
print(df_pred_2025.shape)

## Parte 3 — Predições de risco

### 3.1 Risco domiciliar (por registro)

**Nível de predição: domiciliar.** Já calculado na Parte 2
(`df_pred_2025`); ver `outputs/notebook_final/predicoes_individuais_2025.csv`.

### 3.2 Risco municipal (agregação)

**Nível de predição: municipal.**

In [ ]:
mun_risco = cp.aggregate_municipal(
    df_pred_2025, muni_col="cd_ibge", y_true_col="y_true", y_proba_col="y_proba", min_obs=30
)
mun_risco = mun_risco.sort_values("risco_pred", ascending=False)
mun_risco.to_csv(OUT_DIR / "risco_municipal_2025.csv", index=False)
print(f"Municipios com >=30 obs: {len(mun_risco)}")
mun_risco.head(10)

### 3.3 Mapas coropléticos (risco municipal predito)

**Nível de predição: municipal.** Reusa o shapefile já usado em E5/E9/E10.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

SHP = Path("data/BR_Municipios_2024/BR_Municipios_2024.shp")
gdf = gpd.read_file(SHP)
gdf = gdf[gdf["SIGLA_UF"] == "AL"].copy()
gdf["CD_MUN"] = gdf["CD_MUN"].astype(str).str.replace(r"\.0$", "", regex=True).astype(int)

gdf_risco = gdf.merge(mun_risco, left_on="CD_MUN", right_on="cd_ibge", how="left")

fig, ax = plt.subplots(figsize=(8, 10))
gdf_risco.plot(column="risco_pred", cmap="OrRd", linewidth=0.4, edgecolor="black",
               legend=True, missing_kwds={"color": "lightgrey"}, ax=ax)
ax.axis("off")
ax.set_title("Risco médio predito por município - teste 2025")
plt.tight_layout()
plt.savefig(OUT_DIR / "mapa_risco_municipal_2025.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4 Ranking municipal

**Nível de predição: municipal.**

In [ ]:
ranking = mun_risco[["cd_ibge", "n_obs", "prev_obs", "risco_pred"]].reset_index(drop=True)
ranking.index = ranking.index + 1
ranking.to_csv(OUT_DIR / "ranking_municipal_2025.csv")
ranking.head(15)

## Parte 4 — Validações de robustez espaço-temporal

Cada subseção: objetivo + nível de predição + script de origem (migrado
sem alterar a lógica), código via `common_pipeline`, tabela/figura salva
em `outputs/E{n}/` (mesma convenção usada por `consolidate_tables.py` e
`texto/*.md`/Cap05), interpretação curta.

Cuidado de RAM: `del` + `gc.collect()` entre subseções que carregam
`X_trainval_prepared`/`X_test_prepared` (≈295 MB cada).

### 4.1 E2 — Validação espacial via `GroupKFold` (domiciliar)

Origem: `scripts/E2_spatial_cv.py`. `GroupKFold(n_splits=5)` agrupando
por `d.cd_ibge` sobre `X_trainval_prepared`/`y_trainval` (2024 inteiro).

In [ ]:
from sklearn.model_selection import GroupKFold

OUT_E2 = Path("outputs/E2"); OUT_E2.mkdir(parents=True, exist_ok=True)
# Drop das 6 colunas high-card text (igual a E1/E3), padronizado em 217
# features em 2026-06-26 para nao ter dois numeros de n_features no TCC.
X_trainval_e2 = X_trainval.drop(columns=cp.DROP_HIGH_CARD_TEXT, errors="ignore")
groups = X_trainval_e2["d.cd_ibge"].astype(int).values

gkf = GroupKFold(n_splits=5)
fold_results = []
for fold_idx, (tr_idx, te_idx) in enumerate(gkf.split(X_trainval_e2, y_trainval, groups=groups)):
    X_tr, X_te = X_trainval_e2.iloc[tr_idx], X_trainval_e2.iloc[te_idx]
    y_tr, y_te = y_trainval.iloc[tr_idx], y_trainval.iloc[te_idx]
    X_tr_p = cp.prepare_lgbm_input(cp.cast_numeric_light(X_tr))
    X_te_p = cp.prepare_lgbm_input(X_te, reference_columns=X_tr_p.columns.tolist())
    model = cp.train_lgbm(X_tr_p, y_tr)
    proba = model.predict_proba(X_te_p)[:, 1]
    m = cp.evaluate(y_te, proba)
    m["fold"] = fold_idx + 1
    fold_results.append(m)
    print(f"Fold {fold_idx+1}: AUC-ROC={m['AUC-ROC']:.4f}")

df_e2 = pd.DataFrame(fold_results)
df_e2.to_csv(OUT_E2 / "spatial_cv_metrics.csv", index=False)
print(df_e2[["fold", "AUC-ROC", "AUC-PR", "F2"]].describe())

### 4.2 E11 — Validação espaço-temporal combinada (domiciliar)

Origem: `scripts/E11_spatial_temporal_validation.py`. 5 folds:
treino em municípios A de 2024, teste em municípios B (disjuntos) de 2025.

In [ ]:
OUT_E11 = Path("outputs/E15"); OUT_E11.mkdir(parents=True, exist_ok=True)

# Drop das 6 colunas high-card text (igual a E1/E3), padronizado em 217
# features em 2026-06-26 para nao ter dois numeros de n_features no TCC.
X_trainval_e11 = X_trainval.drop(columns=cp.DROP_HIGH_CARD_TEXT, errors="ignore")
X_test_e11 = X_test.drop(columns=cp.DROP_HIGH_CARD_TEXT, errors="ignore").reindex(columns=X_trainval_e11.columns)

muni_24 = X_trainval_e11["d.cd_ibge"].astype(str).values
muni_25 = X_test_e11["d.cd_ibge"].astype(str).values
munis_unicos = np.unique(np.concatenate([muni_24, muni_25]))

rng = np.random.default_rng(RANDOM_STATE)
folds = np.array_split(rng.permutation(munis_unicos), 5)

results_e15 = []
for k, test_munis in enumerate(folds, start=1):
    train_munis = np.array([m for m in munis_unicos if m not in set(test_munis)])
    mask_tr = np.isin(muni_24, train_munis)
    mask_te = np.isin(muni_25, test_munis)
    X_tr_p = cp.prepare_lgbm_input(cp.cast_numeric_light(X_trainval_e11.loc[mask_tr]))
    X_te_p = cp.prepare_lgbm_input(X_test_e11.loc[mask_te], reference_columns=X_tr_p.columns.tolist())
    model = cp.train_lgbm(X_tr_p, y_trainval[mask_tr])
    proba = model.predict_proba(X_te_p)[:, 1]
    m = cp.evaluate(y_test[mask_te], proba)
    m["fold"] = k
    results_e15.append(m)
    print(f"Fold {k}: AUC-ROC={m['AUC-ROC']:.4f}")

df_e11 = pd.DataFrame(results_e15)
df_e11.to_csv(OUT_E11 / "spatio_temporal_metrics.csv", index=False)
print(df_e11[["fold", "AUC-ROC", "AUC-PR", "F2"]].describe())

### 4.3 E3 — Ablação territorial em 3 níveis A0/A1/A2 (domiciliar)

Origem: `scripts/E3_ablation_3levels.py`. A0=baseline, A1=sem TER,
A2=sem TER+ADM, via `data/anon_outputs/mapeamento_variaveis_inicial.csv`.

In [ ]:
TER_VARS, ADM_VARS = cp.load_ter_adm_vars()

OUT_E3 = Path("outputs/E3"); OUT_E3.mkdir(parents=True, exist_ok=True)
schema = set(X_full_common.columns)
ter_in = TER_VARS & schema
adm_in = ADM_VARS & schema
print(f"TER sobreviventes: {len(ter_in)}/14   ADM sobreviventes: {len(adm_in)}/14")

results_e3 = []
for level, cols_drop in [
    ("A0_baseline", set()),
    ("A1_sem_TER", ter_in),
    ("A2_sem_TER_ADM", ter_in | adm_in),
]:
    X_tr = X_full_common.drop(columns=cols_drop, errors="ignore")
    X_te = X_test_common.drop(columns=cols_drop, errors="ignore")
    X_tr_p = cp.prepare_lgbm_input(cp.cast_numeric_light(X_tr))
    X_te_p = cp.prepare_lgbm_input(X_te, reference_columns=X_tr_p.columns.tolist())
    model = cp.train_lgbm(X_tr_p, y_trainval, scale_pos_weight=1.0)
    proba = model.predict_proba(X_te_p)[:, 1]
    m = cp.evaluate(y_test, proba)
    m["level"] = level
    m["n_features"] = X_tr_p.shape[1]
    results_e3.append(m)
    print(f"{level}: n_feat={m['n_features']}  AUC-ROC={m['AUC-ROC']:.4f}")

df_e3 = pd.DataFrame(results_e3)
df_e3.to_csv(OUT_E3 / "ablation_3levels_notebook_final.csv", index=False)
df_e3

### 4.4 E12 — Evolução de esquema 2023→2024→2025 (domiciliar)

Origem: `scripts/E12_evolucao_esquema.py`. Auditoria estrutural de
headers + impacto preditivo da restrição ao esquema de 2023.

In [ ]:
OUT_E12 = Path("outputs/E16"); OUT_E12.mkdir(parents=True, exist_ok=True)
schemas = {}
for year in [2023, 2024, 2025]:
    df0 = pd.read_csv(DATA_DIR / f"cadunico_{year}_anon.csv", nrows=0)
    schemas[year] = set(df0.columns)
    print(f"{year}: {len(schemas[year])} colunas")

intersect_23_24_25 = schemas[2023] & schemas[2024] & schemas[2025]
print(f"Intersecao 2023∩2024∩2025: {len(intersect_23_24_25)}")

# Drop das 6 colunas high-card text (igual a E1/E3), padronizado em 217
# features em 2026-06-26 para nao ter dois numeros de n_features no TCC.
X_trainval_e12 = X_trainval.drop(columns=cp.DROP_HIGH_CARD_TEXT, errors="ignore")
X_test_e12 = X_test.drop(columns=cp.DROP_HIGH_CARD_TEXT, errors="ignore").reindex(columns=X_trainval_e12.columns)

cols_24 = list(X_trainval_e12.columns)
cols_em_23 = [c for c in cols_24 if c in schemas[2023]]
print(f"Colunas X_trainval em comum com 2023: {len(cols_em_23)}/{len(cols_24)}")

X_tr_p = cp.prepare_lgbm_input(cp.cast_numeric_light(X_trainval_e12[cols_em_23]))
X_te_p = cp.prepare_lgbm_input(X_test_e12[cols_em_23], reference_columns=X_tr_p.columns.tolist())
model_restrito = cp.train_lgbm(X_tr_p, y_trainval)
proba_restrito = model_restrito.predict_proba(X_te_p)[:, 1]
m_restrito = cp.evaluate(y_test, proba_restrito)
m_restrito["n_features"] = len(cols_em_23)
print("Restrito a esquema 2023:", m_restrito)

pd.DataFrame([m_restrito]).to_csv(OUT_E12 / "impacto_restricao_notebook_final.csv", index=False)

In [ ]:
del X_tr_p, X_te_p, model_restrito
gc.collect()
print("Memoria liberada apos E16.")

### 4.5 E9 — Moran's I global e LISA por ciclo do alvo (municipal)

Origem: `scripts/E9_moran_lisa_por_ciclo.py`. Reproduz e estende
`Barbosa2026PT` (Moran's I global já publicado) com classificação LISA
por município.

In [ ]:
from esda.moran import Moran, Moran_Local
from libpysal.weights import Queen

OUT_E9 = Path("outputs/E13"); OUT_E9.mkdir(parents=True, exist_ok=True)
QUADRANT_NAMES = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}
PERMUTATIONS = 999
ALPHA = 0.05
MIN_OBS_PER_MUNI = 30

prev_por_ciclo = {}
for year in [2024, 2025]:
    df_y = pd.read_csv(DATA_DIR / f"cadunico_{year}_anon.csv", engine="pyarrow",
                        usecols=["d.cd_ibge", "d.ind_risco_scl_inseg_alim"])
    df_y = df_y.rename(columns={"d.cd_ibge": "cd_ibge", "d.ind_risco_scl_inseg_alim": "scl_risco"})
    df_y["inseg_alim_bin"] = df_y["scl_risco"].map({1.0: 1, 2.0: 0})
    df_y = df_y.dropna(subset=["inseg_alim_bin", "cd_ibge"])
    df_y["cd_ibge"] = df_y["cd_ibge"].astype(int)
    agg = df_y.groupby("cd_ibge").agg(n_obs=("inseg_alim_bin", "size"),
                                       prev_obs=("inseg_alim_bin", "mean")).reset_index()
    agg = agg[agg["n_obs"] >= MIN_OBS_PER_MUNI]
    prev_por_ciclo[year] = agg

gdf_al = gpd.read_file(SHP)
gdf_al = gdf_al[gdf_al["SIGLA_UF"] == "AL"].copy()
gdf_al["CD_MUN"] = gdf_al["CD_MUN"].astype(str).str.replace(r"\.0$", "", regex=True).astype(int)

moran_por_ciclo = {}
for year, agg in prev_por_ciclo.items():
    gdf_y = gdf_al.merge(agg, left_on="CD_MUN", right_on="cd_ibge", how="left").dropna(subset=["prev_obs"]).reset_index(drop=True)
    w = Queen.from_dataframe(gdf_y, use_index=True); w.transform = "r"
    np.random.seed(RANDOM_STATE)
    moran = Moran(gdf_y["prev_obs"].values, w, permutations=PERMUTATIONS)
    np.random.seed(RANDOM_STATE)
    lisa = Moran_Local(gdf_y["prev_obs"].values, w, permutations=PERMUTATIONS)
    gdf_y["lisa_class"] = np.where(lisa.p_sim < ALPHA, pd.Series(lisa.q).map(QUADRANT_NAMES), "NS")
    gdf_y[["CD_MUN", "NM_MUN", "n_obs", "prev_obs", "lisa_class"]].to_csv(
        OUT_E9 / f"lisa_quadrantes_{year}_notebook_final.csv", index=False)
    moran_por_ciclo[year] = round(float(moran.I), 6)
    print(f"{year}: Moran's I = {moran.I:.4f}  p={moran.p_sim:.4f}")

print("Esperado (Barbosa2026PT / tab:e9_comparacao_pt): 2024=0.1147  2025=0.0967")

### 4.6 E10 — Persistência, emergência e desaparecimento LISA (municipal)

Origem: `scripts/E10_persistencia_lisa.py`. Depende dos CSVs LISA
gerados em 4.6 (ou de `outputs/E13/lisa_quadrantes_*.csv` já existentes).

In [ ]:
SIGNIF = {"HH", "LH", "LL", "HL"}

def classificar_transicao(a, b):
    if a == "NS" and b == "NS":
        return "estavel-NS"
    if a == "NS" and b in SIGNIF:
        return "emergencia"
    if a in SIGNIF and b == "NS":
        return "desaparecimento"
    if a == b:
        return "persistencia"
    return "mudanca"

lisa24 = pd.read_csv(Path("outputs/E13") / "lisa_quadrantes_2024_notebook_final.csv")
lisa25 = pd.read_csv(Path("outputs/E13") / "lisa_quadrantes_2025_notebook_final.csv")
trans = lisa24.merge(lisa25[["CD_MUN", "lisa_class"]], on="CD_MUN", suffixes=("_2024", "_2025"))
trans["transicao"] = trans.apply(
    lambda r: classificar_transicao(r["lisa_class_2024"], r["lisa_class_2025"]), axis=1)

print(trans["transicao"].value_counts())
OUT_E10 = Path("outputs/E14"); OUT_E10.mkdir(parents=True, exist_ok=True)
trans.to_csv(OUT_E10 / "transicoes_lisa_notebook_final.csv", index=False)

In [ ]:
gc.collect()
print("Memoria liberada apos E9/E10.")


### 4.7 E5 — Erro municipal e Moran's I do erro (municipal)

Origem: `scripts/E5_municipal_error.py`. Reusa `df_pred_2025` e
`mun_risco` já calculados na Parte 3.

In [ ]:
OUT_E5 = Path("outputs/E5"); OUT_E5.mkdir(parents=True, exist_ok=True)
mun_erro = mun_risco.copy()
mae = mun_erro["abs_erro"].mean()
me = mun_erro["erro"].mean()
print(f"MAE municipal: {mae:.4f}   ME municipal: {me:.4f}")

gdf_erro = gdf.merge(mun_erro, left_on="CD_MUN", right_on="cd_ibge", how="left").dropna(subset=["erro"]).reset_index(drop=True)
w = Queen.from_dataframe(gdf_erro, use_index=True); w.transform = "r"
np.random.seed(RANDOM_STATE)
moran_erro = Moran(gdf_erro["erro"].values, w, permutations=999)
print(f"Moran's I do erro: {moran_erro.I:.4f}  p={moran_erro.p_sim:.4f}")

pd.DataFrame([{"MAE": mae, "ME": me, "moran_I_erro": moran_erro.I, "p_sim": moran_erro.p_sim}]
             ).to_csv(OUT_E5 / "erro_municipal_notebook_final.csv", index=False)

### 4.8 E7 — SHAP agrupado em 5 categorias (TER/ADM separados) (domiciliar)

Origem: `scripts/E7_shap_5_grupos.py`. Reusa `final_model` (Parte 2) e
decompõe a categoria *Territorial* da co-orientadora em TER puro e ADM.

In [ ]:
manual_group_map = {
    "d.nom_estab_assist_saude_fam": "Territorial", "d.dat_ult_atual_fam": "Temporal",
    "d.qtd_pessoas_domic_fam": "Socioeconomic", "d.vlr_renda_media_fam": "Socioeconomic",
    "d.vlr_renda_total_fam": "Socioeconomic", "d.val_desp_alimentacao_fam": "Socioeconomic",
    "d.val_desp_agua_esgoto_fam": "Socioeconomic", "d.val_desp_energia_fam": "Socioeconomic",
    "d.val_desp_aluguel_fam": "Socioeconomic", "d.cod_ibge_fam": "Territorial",
    "p.cod_ibge_munic_nasc_pessoa": "Territorial", "d.cod_eas_fam": "Territorial",
    "d.cod_material_piso_fam": "Socioeconomic", "d.cod_calcamento_domic_fam": "Socioeconomic",
    "p.cod_curso_frequentou_pessoa_memb": "Socioeconomic", "d.vlr_fam_dia_semana": "Administrative",
    "d.cod_abaste_agua_domic_fam": "Socioeconomic",
}
territorial_keywords = ["ibge", "munic", "uf", "bairro", "territ", "local", "centro_assist",
                          "eas", "estab_assist", "escola", "nom_munic", "cod_munic", "cd_ibge"]
temporal_keywords = ["dat_", "data_", "dta_", "_mes", "_ano", "_dia", "_semana", "cadastro", "atual", "entrevista"]
administrative_keywords = ["cadastro", "entrevista", "responsavel", "formulario", "atend", "posto", "gestao", "controle", "atualizacao"]
socioeconomic_keywords = ["renda", "rfpc", "desp", "qtd_", "comodo", "domic", "agua", "energia",
                            "aluguel", "material", "curso", "escolar", "trabalho", "ocup", "idade"]

def assign_group_keila(f):
    if f in manual_group_map:
        return manual_group_map[f]
    fl = f.lower()
    if any(k in fl for k in territorial_keywords):
        return "Territorial"
    if any(k in fl for k in temporal_keywords):
        return "Temporal"
    if any(k in fl for k in administrative_keywords):
        return "Administrative"
    if any(k in fl for k in socioeconomic_keywords):
        return "Socioeconomic"
    return "Other"

def split_territorial(f):
    if f in TER_VARS:
        return "TER"
    if f in ADM_VARS:
        return "ADM"
    return "TER"

def final_category(f):
    g = assign_group_keila(f)
    if g == "Territorial":
        return split_territorial(f)
    if g == "Socioeconomic":
        return "SOC"
    if g == "Temporal":
        return "TEMP"
    return "OTHER"

X_val_common = X_val.drop(columns=common_drop_cols, errors="ignore").reindex(columns=X_full_common.columns)
sample_size = min(3000, len(X_val_common))
X_val_sample = X_val_common.sample(sample_size, random_state=RANDOM_STATE)
X_val_sample_p = cp.prepare_lgbm_input(X_val_sample, reference_columns=X_full_common.columns.tolist())
X_val_sample_num = X_val_sample_p.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)

shap_raw = final_model.booster_.predict(X_val_sample_num.values, pred_contrib=True)
shap_values = shap_raw[:, :-1]
mean_abs_shap = np.abs(shap_values).mean(axis=0)

df_shap = pd.DataFrame({"feature": X_val_sample_p.columns, "mean_abs_shap": mean_abs_shap})
df_shap["categoria_5cat"] = df_shap["feature"].apply(final_category)

g5 = df_shap.groupby("categoria_5cat").agg(soma=("mean_abs_shap", "sum"), n=("feature", "count")).reset_index()
g5["pct"] = 100 * g5["soma"] / g5["soma"].sum()
g5 = g5.sort_values("pct", ascending=False)
print(g5)

OUT_E7 = Path("outputs/E8_5grupos"); OUT_E7.mkdir(parents=True, exist_ok=True)
g5.to_csv(OUT_E7 / "shap_grouped_5cat_notebook_final.csv", index=False)

In [ ]:
del X_val_common, X_val_sample, X_val_sample_p, X_val_sample_num
gc.collect()
print("Memoria liberada apos E7 (SHAP 5 grupos).")


## Parte 5 — Síntese final

Tabela consolidada E1-E12 com coluna de nível de predição, reunindo os
resultados calculados neste notebook nas Partes 2-4. Os valores de
referência completos (incluindo os experimentos que dependem de
artefatos não recalculados aqui, como E4/E8) permanecem documentados em
`texto/latex_tcc/Cap05/Capitulo5.tex` (`tab:t3_comparativo`).

In [ ]:
sintese = pd.DataFrame([
    {"id": "E1", "nivel": "domiciliar", **{k: round(v, 4) for k, v in metrics_e1.items() if k in ("AUC-ROC", "AUC-PR", "F2")}},
])
for _, row in df_e2.mean(numeric_only=True).to_frame().T.iterrows():
    sintese = pd.concat([sintese, pd.DataFrame([{"id": "E2", "nivel": "domiciliar",
        "AUC-ROC": round(row["AUC-ROC"], 4), "AUC-PR": round(row["AUC-PR"], 4), "F2": round(row["F2"], 4)}])], ignore_index=True)
for _, row in df_e11.mean(numeric_only=True).to_frame().T.iterrows():
    sintese = pd.concat([sintese, pd.DataFrame([{"id": "E11", "nivel": "domiciliar",
        "AUC-ROC": round(row["AUC-ROC"], 4), "AUC-PR": round(row["AUC-PR"], 4), "F2": round(row["F2"], 4)}])], ignore_index=True)

print(sintese)
sintese.to_csv(OUT_DIR / "sintese_consolidada.csv", index=False)